<a href="https://colab.research.google.com/github/Theno1gitty/A-Game-Theory-Simulation-of-the-US-Iran-conflict/blob/main/game_theory_sim_US_Iran.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import random
import pandas as pd
import itertools

random.seed(42)

def get_us_strategy(current_round, iran_history, us_p_defect_after_iran_coop):
    if current_round == 0:
        return "Cooperate"
    else:
        if iran_history[-1] == "Defect":
            if random.random() < 0.75:
                return "Defect"
            else:
                return "Cooperate"
        else:
            if random.random() < us_p_defect_after_iran_coop:
                return "Defect"
            else:
                return "Cooperate"

def get_iran_strategy(current_round, us_history, iran_reserves, iran_p_defect_after_us_coop):
    if iran_reserves < 300:
        return "Cooperate"

    if current_round == 0:
        return "Defect"
    else:
        if us_history[-1] == "Defect":
            if random.random() < 0.80:
                return "Defect"
            else:
                return "Cooperate"
        else:
            if random.random() < iran_p_defect_after_us_coop:
                return "Defect"
            else:
                return "Cooperate"

def run_simulation(max_rounds, iran_cost_multiplier, initial_us_reserves, initial_iran_reserves,
                   us_p_defect_after_iran_coop, iran_p_defect_after_us_coop):
    # print("Starting Geopolitical Escalation Simulation") # Commented out to reduce output during sweep

    us_reserves = initial_us_reserves
    iran_reserves = initial_iran_reserves

    us_history = []
    iran_history = []
    us_reserves_history = []
    iran_reserves_history = []

    for r in range(max_rounds):
        us_reserves_history.append(us_reserves)
        iran_reserves_history.append(iran_reserves)

        us_move = get_us_strategy(r, iran_history, us_p_defect_after_iran_coop)
        iran_move = get_iran_strategy(r, us_history, iran_reserves, iran_p_defect_after_us_coop)

        us_history.append(us_move)
        iran_history.append(iran_move)

        us_cost = 50
        iran_cost = 30

        if us_move == "Defect" and iran_move == "Defect":
            us_cost += 400
            iran_cost += 200
        elif us_move == "Defect" and iran_move == "Cooperate":
            us_cost += 150
            iran_cost += 300
        elif us_move == "Cooperate" and iran_move == "Defect":
            us_cost += (250 * iran_cost_multiplier)
            iran_cost += 50
        else:
            us_cost -= 20
            iran_cost -= 10

        us_reserves -= us_cost
        iran_reserves -= iran_cost

        # print(f"Round {r+1}: US={us_move} | Iran={iran_move} -> Reserves: US={us_reserves}, Iran={iran_reserves})") # Commented out to reduce output during sweep

        if us_move == "Cooperate" and iran_move == "Cooperate":
            # print("\n[TERMINAL STATE] Mutual De-escalation: Both sides wish to exist conflict. Negotiated Settlement achieved.") # Commented out
            us_reserves_history.append(us_reserves)
            iran_reserves_history.append(iran_reserves)
            return "Settlement", us_reserves_history, iran_reserves_history

        if us_reserves <= 0 and iran_reserves <= 0:
            # print("\n[TERMINAL STATE] Mutual Destruction: Extreme cost escalation overshoots both system thresholds.") # Commented out
            us_reserves_history.append(us_reserves)
            iran_reserves_history.append(iran_reserves)
            return "Catastrophe", us_reserves_history, iran_reserves_history
        elif us_reserves <= 0:
            # print("\n[TERMINAL STATE] Iranian Leverage: US hits a critical domestic fiscal or political cost ceiling.") # Commented out
            us_reserves_history.append(us_reserves)
            iran_reserves_history.append(iran_reserves)
            return "Iran Strategic Advantage", us_reserves_history, iran_reserves_history
        elif iran_reserves <= 0:
            # print("\n[TERMINAL STATE] Allied Victory: Iranian power projection or regime cohesion collapses under sustained attrition.") # Commented out
            us_reserves_history.append(us_reserves)
            iran_reserves_history.append(iran_reserves)
            return "Allied Strategic Victory", us_reserves_history, iran_reserves_history

    # print("\n[TERMINAL STATE] Stalemate: Maximum simulation rounds reached without clear resolution.") # Commented out
    us_reserves_history.append(us_reserves)
    iran_reserves_history.append(iran_reserves)
    return "Stalemate", us_reserves_history, iran_reserves_history

#Simulation Parameters. Edit the numbers here to simulate...
MAX_ROUNDS = 15
IRAN_COST_MULTIPLIER = [3, 5, 7]
INITIAL_US_RESERVES = [5000, 7000, 9000, 11000]
INITIAL_IRAN_RESERVES = [1000, 2000, 3000, 4000]
US_P_DEFECT_AFTER_IRAN_COOP = [0.2, 0.4, 0.6, 0.8]
IRAN_P_DEFECT_AFTER_US_COOP = [0.2, 0.4, 0.6, 0.8]

def run_parameter_sweep(
    max_rounds,
    iran_cost_multiplier_values,
    us_p_defect_values,
    iran_p_defect_values,
    initial_us_reserves_values, # New parameter
    initial_iran_reserves_values # New parameter
):
    results = []
    for iran_cost_multiplier, us_p_defect, iran_p_defect, initial_us_reserves, initial_iran_reserves in itertools.product(
        iran_cost_multiplier_values,
        us_p_defect_values,
        iran_p_defect_values,
        initial_us_reserves_values, # Iterate over initial US reserves
        initial_iran_reserves_values # Iterate over initial Iran reserves
    ):
        import sys
        from io import StringIO
        old_stdout = sys.stdout
        sys.stdout = StringIO()

        final_state, us_res_hist_temp, iran_res_hist_temp = run_simulation(
            max_rounds=max_rounds,
            iran_cost_multiplier=iran_cost_multiplier,
            initial_us_reserves=initial_us_reserves,
            initial_iran_reserves=initial_iran_reserves,
            us_p_defect_after_iran_coop=us_p_defect,
            iran_p_defect_after_us_coop=iran_p_defect
        )

        sys.stdout = old_stdout

        results.append({
            'iran_cost_multiplier': iran_cost_multiplier,
            'us_p_defect_after_iran_coop': us_p_defect,
            'iran_p_defect_after_us_coop': iran_p_defect,
            'initial_us_reserves': initial_us_reserves, # Add to results
            'initial_iran_reserves': initial_iran_reserves, # Add to results
            'final_state': final_state,
            'Final US Reserves': us_res_hist_temp[-1],
            'Final Iran Reserves': iran_res_hist_temp[-1]
        })
    df_results = pd.DataFrame(results)
    df_results.to_csv('results.csv', index=False)
    return df_results


print("Running a parameter sweep to analyze different scenarios...")
sweep_results_df = run_parameter_sweep(
    MAX_ROUNDS,
    IRAN_COST_MULTIPLIER,
    US_P_DEFECT_AFTER_IRAN_COOP,
    IRAN_P_DEFECT_AFTER_US_COOP,
    INITIAL_US_RESERVES, # Pass the list of initial US reserves
    INITIAL_IRAN_RESERVES # Pass the list of initial Iran reserves
)
print("Parameter sweep completed. Displaying the first few results:")
display(sweep_results_df.head(1500))

Running a parameter sweep to analyze different scenarios...
Parameter sweep completed. Displaying the first few results:


,iran_cost_multiplier,us_p_defect_after_iran_coop,iran_p_defect_after_us_coop,initial_us_reserves,initial_iran_reserves,final_state,Final US Reserves,Final Iran Reserves
0,3,0.2,0.2,5000,1000,Settlement,2820,210
1,3,0.2,0.2,5000,2000,Settlement,1770,-10
2,3,0.2,0.2,5000,3000,Settlement,1670,2280
3,3,0.2,0.2,5000,4000,Settlement,3170,3490
4,3,0.2,0.2,7000,1000,Settlement,6170,900
...,...,...,...,...,...,...,...,...
763,7,0.8,0.8,9000,4000,Iran Strategic Advantage,-200,2430
764,7,0.8,0.8,11000,1000,Allied Strategic Victory,6550,-50
765,7,0.8,0.8,11000,2000,Allied Strategic Victory,6100,-120
766,7,0.8,0.8,11000,3000,Settlement,2420,1970
